# 7.23 選了 high，就是固定多想幾秒嗎？


同樣只回一個數字，「2+3 是多少」選 low 就能很快答 5。換成一串會互相改寫的訂單事件，例如先有 2 件、再加 3 件、接著改成 1 件、再加 4 件、再加 2 件，中間的「改成」會推翻前面的數量，就算最後只回一個數字，也得逐步處理更多事件。服務上的**思考投入**（thinking effort，也叫 reasoning effort），是使用者對解題資源的偏好。它值得和 [7.19](https://birdhackor.github.io/tiny-perceptron-vlm/7.19.html) 的長度上限一起學：我們既要控制能放多少材料，也要控制願意花多少工作處理材料。

先分清三件事。**請求檔位**是傳入的 low、medium、high 等名稱；**執行設定**是服務把名稱換成的規則或預算；**實際耗用**才是這題真的用了幾個推理 token、幾輪、幾次工具或多少時間。high 通常允許或鼓勵多做工作，但不必每題都用完同樣的額度。不同模型接受的名稱、預設值及硬上限也不同，不能把這三欄合成一個數字。

下面用一個玩具規則把三欄分開看：假設每個檔位對應「最多處理幾個事件」的執行設定，處理一個事件算一輪。這是為了教學編的規則，不代表任何服務的內部做法。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
limits = {"low": 2, "medium": 3, "high": 5}  # 假設的執行設定：每個檔位最多處理幾個事件


def run(events, effort):
    count, rounds = 0, 0
    for action, amount in events:
        if rounds == limits[effort]:
            break
        count = amount if action == "改成" else count + amount
        rounds += 1
    return count, rounds


tasks = {"2+3": [("加", 2), ("加", 3)], "訂單": [("加", 2), ("加", 3), ("改成", 1), ("加", 4), ("加", 2)]}
for name, events in tasks.items():
    for effort in ["low", "medium", "high"]:
        answer, rounds = run(events, effort)
        print(f"{name}｜請求 {effort}｜允許 {limits[effort]} 輪｜實際 {rounds} 輪｜答案 {answer}")

`run` 逐個處理事件：「加」就把數量加上去，「改成」就直接蓋掉原本的數量，`rounds += 1` 把用掉的輪數加 1；處理的輪數到了這個檔位允許的上限，就 `break` 停下，回傳目前的數量與用掉的輪數。2+3 只有兩個事件，所以請求 high 也只用 2 輪，不會為了湊滿 5 輪而多做。訂單題有五個事件：low 處理完前兩個就停，答出 5；medium 停在「改成 1」之後，答出 1；high 用滿 5 輪，才得到正確的 7。六行的答案都只有一個數字，工作量卻不同。

最後答案一樣短，工作量可以不同。反過來，要求「寫長一點」可能只增加說明，沒有增加核對。輸出的詳略、抽樣溫度、上下文視窗的容量、候選答案的份數與共用區塊反覆運算的輪數，各自控制不同的部分；溫度改的是選字機率，不能當成思考時間的旋鈕。共用區塊如何反覆運算，會在 [14.11](https://birdhackor.github.io/tiny-perceptron-vlm/14.11.html) 拆開。

實際的服務也提供這類設定：有的讓模型依題目自己決定想多久，同時也讓使用者指定偏好，兩者可以並存。可用的名稱、範圍與預設值依模型版本而定，不能把本課的 low／medium／high 直接送給所有 API；各家文件的說法見補充。

練習替程式的輸出多加「耗時」一欄，假設每輪 1 秒。訂單題的 high 最慢；只憑這張表，可以判斷服務內部用了哪種架構嗎？不行：生成更多中間 token、呼叫工具，或讓同一個區塊多跑幾輪（[14.11](https://birdhackor.github.io/tiny-perceptron-vlm/14.11.html)），都可能增加耗時。介面與耗時能幫我們管理資源，卻不能拿來推斷模型內部的架構。

<details>
<summary>補充：各家服務的設定與來源</summary>

這是截至 2026-10-16 官方文件的介面例子，不需付費呼叫也能讀完本課。OpenAI 的 reasoning effort 依模型有不同的可用值，文件也說明模型會依題目調整實際的思考量；Claude 的 adaptive thinking 讓模型決定是否思考、思考多少，effort 用來引導取捨；Gemini 提供依模型而異的 thinking level／budget 與動態模式。可用選項仍要查實際的模型版本。

來源：[OpenAI reasoning](https://developers.openai.com/api/docs/guides/reasoning)、[Claude thinking](https://platform.claude.com/docs/en/build-with-claude/thinking-steering-and-cost)、[Gemini thinking](https://ai.google.dev/gemini-api/docs/thinking)。

</details>
